# Triton Neurotech — EMG BCI Workshop

Welcome! In this notebook you'll work through the core ideas behind reading and processing EMG (electromyography) signals from a MindRove armband.

**By the end you'll have:**
- Streamed real (or simulated) EMG data
- Plotted the raw signal
- Built a simple signal processing pipeline
- Made a threshold detector that fires on muscle contractions

---
## Prerequisites

Run this before anything else:

In [ ]:
# Install dependencies if needed
# %pip install mindrove numpy matplotlib pylsl scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.facecolor'] = '#0b0f1a'
plt.rcParams['figure.facecolor'] = '#0b0f1a'
plt.rcParams['axes.edgecolor'] = '#2a3a5e'
plt.rcParams['text.color'] = '#e8ecf8'
plt.rcParams['axes.labelcolor'] = '#e8ecf8'
plt.rcParams['xtick.color'] = '#5a6a94'
plt.rcParams['ytick.color'] = '#5a6a94'
plt.rcParams['grid.color'] = '#1e2a3e'
plt.rcParams['grid.alpha'] = 0.6

print('Imports OK!')

---
## Part 1 — Connect to the Armband

The `EMGInput` class from `emg_input.py` tries backends in this order:
1. **LSL** — MindRove Connect app is streaming (recommended)
2. **Direct SDK** — connect to the armband Wi-Fi directly
3. **Synthetic** — generated fake signal (works without hardware)

### Hardware path
1. Connect to the armband's Wi-Fi
2. Open **MindRove Connect** → click **Start LSL Stream**
3. Run the cell below

### No hardware?
Set `USE_SYNTHETIC = True` below — you'll get a simulated signal that still lets you do all the exercises.

In [ ]:
import sys
sys.path.insert(0, '.')  # make sure local modules are importable

from emg_input import EMGInput

USE_SYNTHETIC = False   # <-- set True if no hardware available

emg = EMGInput(synthetic=USE_SYNTHETIC)
emg.start()

print(f'Backend  : {emg.backend}')
print(f'Channels : {emg.n_channels}')
print(f'Rate     : {emg.sampling_rate} Hz')

---
## Part 2 — Read and Plot Raw Data

`emg.peek(n)` returns the last `n` samples **without consuming them** — good for display.

Data shape is `(n_channels, n_samples)`, values in **microvolts (µV)**.

In [ ]:
# Grab 1 second of data and plot all channels
time.sleep(0.5)  # let the buffer fill

n_samples = emg.sampling_rate  # 1 second worth
data = emg.peek(n_samples)     # shape: (n_channels, n_samples)

print(f'Data shape: {data.shape}   (channels x samples)')

t = np.linspace(0, 1, n_samples)
colors = ['#4d9fff', '#00d26e', '#ffd232', '#ff80ab']

fig, axes = plt.subplots(emg.n_channels, 1, sharex=True,
                          figsize=(14, 2.5 * emg.n_channels))
if emg.n_channels == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.plot(t, data[i], color=colors[i % len(colors)], linewidth=0.8)
    ax.set_ylabel(f'CH{i+1} (µV)', fontsize=10)
    ax.grid(True)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Raw EMG — 1 second', fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 3 — Signal Processing

Raw EMG is a bipolar AC signal centered around 0 V.  
To detect muscle activation, we need to turn it into a smooth **envelope**.

The pipeline used in this project:

```
raw signal  →  DC removal  →  rectify (abs)  →  moving average  →  activation scalar
```

### 3a — DC Removal

Electrodes can have a slow DC offset drift. We remove it with a slow IIR filter:

In [ ]:
# Example: DC removal with IIR baseline tracker
# alpha close to 1 = very slow tracker (keeps slow drift, not fast EMG)

CHANNEL = 0   # which channel to analyze (change to whichever has the best signal)

raw = data[CHANNEL].copy()
alpha = 0.998

baseline = np.zeros_like(raw)
b = 0.0
for i, v in enumerate(raw):
    b = alpha * b + (1 - alpha) * v
    baseline[i] = b

dc_removed = raw - baseline

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)
ax1.plot(t, raw, color='#4d9fff', linewidth=0.8, label='Raw')
ax1.plot(t, baseline, color='#ff8040', linewidth=1.2, linestyle='--', label='DC baseline')
ax1.legend(); ax1.set_ylabel('µV'); ax1.grid(True)

ax2.plot(t, dc_removed, color='#00d26e', linewidth=0.8, label='DC removed')
ax2.legend(); ax2.set_ylabel('µV'); ax2.set_xlabel('Time (s)'); ax2.grid(True)

fig.suptitle('DC Removal')
plt.tight_layout()
plt.show()

### 3b — Rectification + Moving Average

Rectifying (taking `abs()`) makes all values positive.  
A moving average then smooths this into an **envelope**.

In [ ]:
# Rectify
rectified = np.abs(dc_removed)

# Moving average (150 ms window)
smooth_ms = 150
win = max(1, int(emg.sampling_rate * smooth_ms / 1000))
kernel = np.ones(win) / win
envelope = np.convolve(rectified, kernel, mode='same')

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(14, 7))

axes[0].plot(t, dc_removed, color='#4d9fff', linewidth=0.8)
axes[0].set_ylabel('DC removed (µV)'); axes[0].grid(True)

axes[1].plot(t, rectified, color='#ffd232', linewidth=0.8)
axes[1].set_ylabel('Rectified (µV)'); axes[1].grid(True)

axes[2].plot(t, envelope, color='#00d26e', linewidth=1.5)
axes[2].set_ylabel(f'Envelope ({smooth_ms} ms avg)'); axes[2].grid(True)
axes[2].set_xlabel('Time (s)')

fig.suptitle('Signal Processing Pipeline')
plt.tight_layout()
plt.show()

### 3c — Using the built-in processor

The `EMGProcessor` class does all of the above in real-time:

In [ ]:
from signal_processing import EMGProcessor

proc = EMGProcessor(sampling_rate=emg.sampling_rate, smooth_ms=150, channel=CHANNEL)

# Feed new samples and read the activation scalar
new_samples = emg.pull()     # consume new samples since last call
proc.push(new_samples)       # update processor
activation = proc.activation()  # single µV value

print(f'Current activation: {activation:.1f} µV')
print('(flex your muscle and re-run this cell to see a higher value)')

---
## Part 4 — Your Turn: Build a Simple Threshold Detector

**Exercise:** Complete the function below. It should return `True` when the activation crosses a threshold, with a simple cooldown so it doesn't fire multiple times on one contraction.

Hints:
- `activation > threshold` checks if the muscle is active
- Use `time.monotonic()` to track when the last trigger happened
- A `cooldown_s` of ~0.3 seconds prevents rapid re-firing

In [ ]:
# ── YOUR CODE HERE ──────────────────────────────────────────────────────────

class SimpleDetector:
    def __init__(self, threshold: float = 50.0, cooldown_s: float = 0.3):
        self.threshold = threshold
        self.cooldown_s = cooldown_s
        self._last_trigger = 0.0

    def update(self, activation: float) -> bool:
        """
        Returns True once per muscle contraction.
        
        TODO: implement this!
        Steps:
          1. Check if activation is above self.threshold
          2. Check that enough time has passed since self._last_trigger
          3. If both conditions met: update self._last_trigger and return True
          4. Otherwise return False
        """
        # ↓↓ write your code below ↓↓
        pass

# ── Test your detector ───────────────────────────────────────────────────────
detector = SimpleDetector(threshold=30.0, cooldown_s=0.3)

# Simulate a series of activation values
test_activations = [5, 8, 10, 60, 65, 70, 60, 20, 10, 5, 80, 70, 8, 5]
triggers = []

for val in test_activations:
    fired = detector.update(float(val))
    triggers.append(fired)

print('Activations:', test_activations)
print('Triggers:   ', [int(x) for x in triggers])
print()
# Expected: should fire once near the 60/65/70 peak, and once near the 80 peak
# (exact positions depend on your cooldown timing and test speed)

<details>
<summary>Click to see a reference solution</summary>

```python
def update(self, activation: float) -> bool:
    now = time.monotonic()
    if activation > self.threshold and (now - self._last_trigger) >= self.cooldown_s:
        self._last_trigger = now
        return True
    return False
```
</details>

---
## Part 5 — Live Activation Monitor

This cell polls the armband for 10 seconds and prints your activation level in real-time.  
**Flex and relax your arm** to see the values change.

In [ ]:
from IPython.display import clear_output

MONITOR_SECONDS = 10
THRESHOLD = 40.0  # µV — adjust based on your signal

proc2 = EMGProcessor(sampling_rate=emg.sampling_rate, smooth_ms=150, channel=CHANNEL)
detector2 = SimpleDetector(threshold=THRESHOLD, cooldown_s=0.3)

trigger_count = 0
start = time.monotonic()

print(f'Monitoring for {MONITOR_SECONDS}s — flex your arm to trigger!')
print(f'Threshold: {THRESHOLD} µV\n')

while (elapsed := time.monotonic() - start) < MONITOR_SECONDS:
    samples = emg.pull()
    proc2.push(samples)
    act = proc2.activation()

    # Only update display ~20 fps
    bar_len = int(min(act / 5, 40))
    bar = '█' * bar_len + '░' * (40 - bar_len)
    marker = '|' if act >= THRESHOLD else ' '
    fired = detector2.update(act)
    if fired:
        trigger_count += 1

    clear_output(wait=True)
    print(f'Monitoring for {MONITOR_SECONDS}s — flex your arm to trigger!')
    print(f'Threshold: {THRESHOLD} µV\n')
    print(f'  [{bar}]{marker}  {act:6.1f} µV')
    print(f'  Triggers: {trigger_count}   |   Time: {elapsed:.1f} s')
    time.sleep(0.05)

print(f'\nDone! Total triggers in {MONITOR_SECONDS}s: {trigger_count}')

---
## Part 6 — Calibration

Instead of guessing the threshold, we can **measure** it.  
Record what your arm looks like at rest, then while flexing — the threshold goes in between.

Run the cell below. You'll have 3 seconds to relax, then 3 seconds to flex.

In [ ]:
from calibration import Calibration

cal = Calibration()
proc3 = EMGProcessor(sampling_rate=emg.sampling_rate, smooth_ms=150, channel=CHANNEL)

WINDOW_S = 3

def record_window(label, duration_s):
    """Collect activation values over a window and return them."""
    activations = []
    end = time.monotonic() + duration_s
    while time.monotonic() < end:
        samples = emg.pull()
        proc3.push(samples)
        activations.append(proc3.activation())
        time.sleep(0.02)
    return np.array(activations)

print('=== CALIBRATION ===')
print()
print(f'RELAX your arm completely for {WINDOW_S} seconds...')
time.sleep(0.5)
rest_vals = record_window('REST', WINDOW_S)
cal.record_rest(rest_vals)
print(f'  Rest done.  Mean: {cal.rest_mean:.1f} µV')

print()
print(f'Now FLEX repeatedly for {WINDOW_S} seconds...')
time.sleep(0.5)
flex_vals = record_window('FLEX', WINDOW_S)
cal.record_flex(flex_vals)
print(f'  Flex done.  Mean: {cal.flex_mean:.1f} µV')

threshold = cal.compute_threshold()

print()
print('=== RESULTS ===')
print(cal.summary())
print()
print(f'Use threshold = {threshold:.1f} µV in your detector')

In [ ]:
# Plot the calibration data
fig, ax = plt.subplots(figsize=(14, 4))

t_rest = np.linspace(0, WINDOW_S, len(rest_vals))
t_flex = np.linspace(WINDOW_S, 2 * WINDOW_S, len(flex_vals))

ax.plot(t_rest, rest_vals, color='#4d9fff', linewidth=1, label='Rest')
ax.plot(t_flex, flex_vals, color='#00d26e', linewidth=1, label='Flex')
ax.axhline(threshold, color='#ff4848', linewidth=1.5, linestyle='--',
           label=f'Threshold = {threshold:.1f} µV')
ax.axvline(WINDOW_S, color='#5a6a94', linewidth=1, linestyle=':')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Activation (µV)')
ax.set_title('Calibration: Rest vs Flex')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

---
## Part 7 — Wire It All Together

Now you have all the pieces. This is what `main.py` does internally:

1. `EMGInput` — streams raw data from the armband
2. `EMGProcessor` — turns the stream into an activation scalar
3. `Calibration` — provides a data-driven threshold
4. `TriggerController` — converts activations → clean trigger events
5. Game / reaction loop reacts to triggers

Run `python main.py` from the terminal to play the full game with your muscles!

In [ ]:
# Verify the full pipeline works end-to-end
from controller import TriggerController

proc_final = EMGProcessor(sampling_rate=emg.sampling_rate, smooth_ms=150, channel=CHANNEL)
ctrl = TriggerController(threshold=threshold, refractory_ms=300)

print('Full pipeline check:')
print(f'  EMGInput    backend={emg.backend}, channels={emg.n_channels}')
print(f'  EMGProcessor channel={CHANNEL}, smooth_ms=150')
print(f'  TriggerController threshold={threshold:.1f} µV, refractory=300 ms')
print()

# Read one frame
samples = emg.pull()
proc_final.push(samples)
act = proc_final.activation()
fired = ctrl.update(act)

print(f'  Current activation : {act:.1f} µV')
print(f'  Trigger fired      : {fired}')
print()
print('All good! Run `python main.py` to play the full game.')

---
## Bonus — Challenge Ideas

If you've finished the main exercises, try one of these:

**1. Multi-channel max**  
Instead of using a single channel, take the `max()` activation across all channels. Does this make detection more reliable?

**2. Gesture detection**  
The armband has 4 channels. Can you detect the *direction* of a motion (e.g., flex vs. extend) by comparing channels?

**3. Smooth the threshold**  
Instead of a fixed threshold, try adapting it slowly over time (exponential moving average of the recent max).

**4. Frequency analysis**  
EMG signal is roughly 20–500 Hz. Use `np.fft.rfft` to plot the frequency spectrum of your signal during rest vs. flex. What changes?

In [ ]:
# Bonus: FFT of rest vs flex
# Capture two windows manually and compare spectra

SR = emg.sampling_rate
N = SR  # 1 second
freqs = np.fft.rfftfreq(N, d=1.0/SR)

# Grab a window now (rest or flex depending on what you're doing)
window = emg.peek(N)[CHANNEL]
spectrum = np.abs(np.fft.rfft(window - np.mean(window)))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(freqs, spectrum, color='#ba68ff', linewidth=0.8)
ax.set_xlim(0, 500)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Amplitude')
ax.set_title('EMG Frequency Spectrum (current window)')
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Clean up — stop the EMG stream when done
emg.stop()
print('Stream stopped.')